# NASAID SQLite vs Shapefile Validation

This notebook samples 10-20 coordinates from the legacy NASAID shapefile and compares IDs against the SQLite-backed `get_nasaid(lon, lat)` lookup.

Goal: verify the fallback datastore returns the same NASA IDs as the source data.

In [1]:
import sys
from pathlib import Path
import sqlite3
import geopandas as gpd
import pandas as pd

# Ensure project root is importable from notebooks/
project_root = Path.cwd().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from frontend.backend.utils.nasaid_lookup import get_nasaid, DATASTORE_PATH

shp_path = project_root / "frontend" / "sample_data" / "nasaid" / "five_arc_land2_nasa.shp"

print(f"Project root: {project_root}")
print(f"Shapefile path: {shp_path}")
print(f"SQLite datastore: {DATASTORE_PATH}")
print(f"Datastore exists: {DATASTORE_PATH.exists()}")

Project root: C:\Users\itswa\Desktop\ABE\uf-chirps-zarr
Shapefile path: C:\Users\itswa\Desktop\ABE\uf-chirps-zarr\frontend\sample_data\nasaid\five_arc_land2_nasa.shp
SQLite datastore: C:\Users\itswa\Desktop\ABE\uf-chirps-zarr\frontend\backend\data\nasaid_lookup.sqlite
Datastore exists: True


In [2]:
# Load source shapefile and determine the comparison columns
gdf = gpd.read_file(shp_path)

cols_lower = {c.lower(): c for c in gdf.columns}
id_col = cols_lower.get("nasapid") or cols_lower.get("nasaid") or cols_lower.get("id")

if id_col is None:
    raise ValueError("No NASA ID column found in shapefile")

if "lonnp" in cols_lower and "latnp" in cols_lower:
    lon_col, lat_col = cols_lower["lonnp"], cols_lower["latnp"]
elif "longitude" in cols_lower and "latitude" in cols_lower:
    lon_col, lat_col = cols_lower["longitude"], cols_lower["latitude"]
else:
    lon_col, lat_col = None, None

print(f"Rows in source shapefile: {len(gdf)}")
print(f"ID column: {id_col}")
print(f"Coordinate columns used: {lon_col}, {lat_col}")

Rows in source shapefile: 2299097
ID column: nasapid
Coordinate columns used: LonNP, LatNP


In [3]:
# Compare 20 sampled coordinates: shapefile ID vs SQLite-backed get_nasaid
sample_n = min(20, len(gdf))
sample_df = gdf.sample(n=sample_n, random_state=42).copy()

records = []
for _, row in sample_df.iterrows():
    expected = str(int(row[id_col])) if pd.notna(row[id_col]) else None

    if lon_col and lat_col:
        lon = float(row[lon_col])
        lat = float(row[lat_col])
    else:
        lon = float(row.geometry.x)
        lat = float(row.geometry.y)

    actual = get_nasaid(lon=lon, lat=lat)
    records.append(
        {
            "expected_nasaid": expected,
            "actual_nasaid": actual,
            "match": expected == actual,
            "lon": lon,
            "lat": lat,
        }
    )

result_df = pd.DataFrame(records)
match_count = int(result_df["match"].sum())

print(f"Compared {sample_n} sampled coordinates")
print(f"Matches: {match_count}/{sample_n}")
print(f"Mismatches: {sample_n - match_count}/{sample_n}")

if match_count < sample_n:
    print("\nMismatched rows:")
    display(result_df[~result_df["match"]])
else:
    print("All sampled coordinates match.")

print("\nSample comparison table:")
display(result_df.head(20))

Compared 20 sampled coordinates
Matches: 20/20
Mismatches: 0/20
All sampled coordinates match.

Sample comparison table:


,expected_nasaid,actual_nasaid,match,lon,lat
0,216826,216826,True,-127.25,60.75
1,177742,177742,True,130.75,33.25
2,181646,181646,True,-77.25,36.25
3,224712,224712,True,-144.25,66.25
4,121160,121160,True,-80.25,-5.75
5,165289,165289,True,24.25,24.75
6,219330,219330,True,44.75,62.25
7,192846,192846,True,122.75,43.75
8,218333,218333,True,-93.75,61.75
9,183354,183354,True,56.75,37.25
